# Type 2 Diabetes — Exploratory Data Analysis

**Dataset:** `research_grade_type2_diabetes_dataset_v3.csv`  
**Target:** `stage` — diabetes progression stage  
**Goal:** Understand data quality, distribution of the target, and identify key clinical features that drive disease progression.

---

## Research Questions

1. **Target structure** — Is `stage` continuous or ordinal? Skewed? This determines whether we treat the problem as regression or ordinal classification.  
2. **Clinical drivers** — Which biomarkers (glucose, BMI, HbA1c, etc.) have the strongest monotonic relationship with disease stage?  
3. **Non-linearity** — Do Pearson and Spearman correlations diverge? If so, tree-based models are likely necessary.  
4. **Data quality** — Missing values, duplicates, physiologically implausible outliers?  
5. **Feature redundancy** — Are any predictors highly inter-correlated (multicollinearity risk for linear models)?

---

**Sections:**
1. Setup & Data Loading  
2. Initial Inspection  
3. Data Quality — Missingness, Duplicates, Outliers  
4. Target Variable Analysis  
5. Numeric Feature Distributions  
6. Categorical Feature Analysis  
7. Correlation Analysis  
8. Feature–Target Relationships  
9. Summary & Modelling Plan  


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid', palette='muted')

SEED   = 42
TARGET = 'stage'
np.random.seed(SEED)

df = pd.read_csv('Data/research_grade_type2_diabetes_dataset_v3.csv')
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

## 2. Initial Inspection

First look at column names, dtypes, and summary statistics. Pay attention to:
- Whether `stage` is integer (ordinal classes) or float (continuous)
- Any columns with unexpected dtypes (e.g. numeric stored as object)
- Range of clinical values — physiologically implausible values are a data quality flag


In [ ]:
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe().T

## 3. Data Quality

### 3.1 Missingness

Decision rule:
- **< 5%** → median / mode imputation  
- **5–30%** → impute + add binary missingness indicator flag  
- **> 30%** → consider dropping the column


In [ ]:
missing = pd.DataFrame({
    'count': df.isnull().sum(),
    'pct':   df.isnull().mean() * 100
}).query('count > 0').sort_values('pct', ascending=False)

print(f'Columns with missing values: {len(missing)} of {df.shape[1]}')

if len(missing) > 0:
    missing['action'] = missing['pct'].apply(
        lambda p: 'impute (median/mode)' if p < 5
                  else 'impute + flag' if p < 30
                  else 'consider dropping'
    )
    print(missing.to_string())

    plot_df = missing.reset_index().rename(columns={'index': 'column'})
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.barplot(data=plot_df, x='pct', y='column', color='steelblue', ax=ax)
    ax.axvline(5,  color='orange', linestyle='--', linewidth=1.2, label='5% threshold')
    ax.axvline(30, color='red',    linestyle='--', linewidth=1.2, label='30% threshold')
    ax.set_xlabel('Missing (%)')
    ax.set_ylabel('')
    ax.set_title('Missingness by Column')
    ax.legend()
    sns.despine()
    plt.tight_layout()
    plt.show()
else:
    print('No missing values.')

### 3.2 Duplicates

In [ ]:
n_dups = df.duplicated().sum()
print(f'Full duplicate rows: {n_dups} ({n_dups / len(df) * 100:.2f}%)')
if n_dups > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'Removed. New shape: {df.shape}')

### 3.3 Outliers

**3×IQR fence** used — conservative threshold appropriate for clinical data where extreme values may be physiologically real (e.g. very high glucose) rather than entry errors. Flagged for review, not automatically removed.


In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()

outlier_report = []
for col in num_cols:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lo, hi = Q1 - 3 * IQR, Q3 + 3 * IQR
    n_out = ((df[col] < lo) | (df[col] > hi)).sum()
    if n_out > 0:
        outlier_report.append({
            'column': col, 'n_outliers': n_out,
            'pct': round(n_out / len(df) * 100, 2),
            'lower_fence': round(lo, 3), 'upper_fence': round(hi, 3)
        })

if outlier_report:
    out_df = pd.DataFrame(outlier_report).sort_values('pct', ascending=False)
    print('Outliers (3×IQR) — flagged, not removed:')
    print(out_df.to_string(index=False))
else:
    print('No outliers at 3×IQR threshold.')

## 4. Target Variable Analysis

`stage` is the diabetes progression stage. Key questions:
- **Continuous vs ordinal?** Integer values with few unique levels → ordinal classification. Float with many values → regression.  
- **Skewness** — affects choice of loss function. Heavy right-skew → log-transform target or use MAE-based loss.  
- **Class balance** (if ordinal) — severe imbalance requires stratified splits and appropriate metrics (weighted F1 / macro AUC).


In [ ]:
print(f'Unique values of {TARGET}: {df[TARGET].nunique()}')
print(df[TARGET].value_counts().sort_index())
print()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram with KDE
sns.histplot(df[TARGET], bins=50, kde=True, color='steelblue',
             edgecolor='white', ax=axes[0])
axes[0].axvline(df[TARGET].mean(),   color='red',    linestyle='--',
                label=f'Mean={df[TARGET].mean():.2f}')
axes[0].axvline(df[TARGET].median(), color='orange', linestyle='--',
                label=f'Median={df[TARGET].median():.2f}')
axes[0].set_title(f'{TARGET} — Distribution')
axes[0].set_xlabel(TARGET)
axes[0].legend(fontsize=9)

# Boxplot
sns.boxplot(y=df[TARGET], color='steelblue', width=0.4, ax=axes[1])
axes[1].set_title(f'{TARGET} — Boxplot')
axes[1].set_ylabel(TARGET)

sns.despine()
plt.tight_layout()
plt.show()

skew = df[TARGET].skew()
kurt = df[TARGET].kurt()
print(f'Skewness : {skew:.3f}  {"→ right-skewed" if skew>1 else "→ left-skewed" if skew<-1 else "→ approx. symmetric"}')
print(f'Kurtosis : {kurt:.3f}  {"→ heavy tails" if kurt>3 else ""}')
print(f'Min / Max: {df[TARGET].min()} / {df[TARGET].max()}')

## 5. Numeric Feature Distributions

Grid of histograms for all numeric columns. Skewness annotated on each — right-skewed features (skew > 1) are candidates for log-transform before linear modelling.


In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
print(f'Numeric columns: {len(num_cols)}')

n_cols = 3
n_rows = (len(num_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(df[col].dropna(), bins=40, kde=True, color='steelblue',
                 edgecolor='white', alpha=0.8, ax=axes[i])
    skew = df[col].skew()
    flag = ' ⚠' if abs(skew) > 1 else ''
    axes[i].set_title(f'{col}\nskew={skew:.2f}{flag}', fontsize=9)
    axes[i].set_xlabel('')
    axes[i].tick_params(labelsize=7)
    sns.despine(ax=axes[i])

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numeric Feature Distributions  (⚠ = |skew| > 1, consider log-transform)', y=1.01)
plt.tight_layout()
plt.show()

## 6. Categorical Feature Analysis

Cardinality reported per column. High-cardinality columns (> 20 unique values) are flagged — they require target-encoding or should be dropped before modelling to avoid dimensionality explosion from one-hot encoding.


In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns: {len(cat_cols)}')

for col in cat_cols:
    n_unique = df[col].nunique()
    flag = '  ⚠ HIGH CARDINALITY' if n_unique > 20 else ''
    print(f'\n{col}: {n_unique} unique values{flag}')
    print(df[col].value_counts().head(10).to_string())

    if n_unique <= 20:
        order = df[col].value_counts().head(10).index
        fig, ax = plt.subplots(figsize=(8, 3))
        sns.countplot(data=df, y=col, order=order, color='steelblue', ax=ax)
        ax.set_title(f'{col} — Value Counts')
        ax.set_xlabel('Count')
        ax.set_ylabel('')
        sns.despine()
        plt.tight_layout()
        plt.show()

## 7. Correlation Analysis

**Pearson** — linear association. **Spearman** — monotonic association, robust to outliers and non-normality. For clinical ordinal data, Spearman is preferred.

The `divergence` column (`|Spearman| − |Pearson|`) highlights features with non-linear signal — these benefit most from tree-based models.


In [ ]:
corr = df[num_cols].corr(method='pearson')

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax, annot_kws={'size': 7}
)
ax.set_title('Pearson Correlation Matrix (lower triangle)')
plt.tight_layout()
plt.show()

### 7.1 Feature Correlation with Target

In [ ]:
feat_cols = [c for c in num_cols if c != TARGET]
target_corr = pd.DataFrame({
    'pearson' : [df[c].corr(df[TARGET], method='pearson')  for c in feat_cols],
    'spearman': [df[c].corr(df[TARGET], method='spearman') for c in feat_cols],
}, index=feat_cols)
target_corr['abs_spearman'] = target_corr['spearman'].abs()
target_corr['divergence']   = target_corr['spearman'].abs() - target_corr['pearson'].abs()
target_corr = target_corr.sort_values('abs_spearman', ascending=False)

print(f'Feature correlations with {TARGET}:')
print(target_corr[['pearson', 'spearman', 'divergence']].to_string())

plot_df = target_corr.reset_index().rename(columns={'index': 'feature'})
plot_df['color'] = plot_df['spearman'].apply(lambda x: 'positive' if x >= 0 else 'negative')

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(
    data=plot_df.sort_values('spearman'),
    x='spearman', y='feature',
    hue='color',
    palette={'positive': 'steelblue', 'negative': 'coral'},
    legend=False, ax=ax
)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f'Spearman Correlation with {TARGET}')
ax.set_xlabel('Spearman ρ')
ax.set_ylabel('')
sns.despine()
plt.tight_layout()
plt.show()

## 8. Feature–Target Scatter Plots

Top-6 features by |Spearman ρ|. Linear trend overlaid — curvature signals non-linearity; fan shape signals heteroskedasticity (error variance grows with predicted value). p-value shown for each correlation.


In [ ]:
top_features = target_corr.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(top_features):
    valid = df[[col, TARGET]].dropna()
    # Sample for speed if large
    sample = valid.sample(min(3000, len(valid)), random_state=42)
    sns.regplot(
        data=sample, x=col, y=TARGET,
        scatter_kws={'alpha': 0.25, 's': 10, 'color': 'steelblue'},
        line_kws={'color': 'red', 'linewidth': 1.5, 'linestyle': '--'},
        ax=axes[i]
    )
    rho, p = spearmanr(valid[col], valid[TARGET])
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'
    axes[i].set_title(f'{col}\nρ={rho:.3f}  {sig}', fontsize=9)
    axes[i].set_xlabel(col, fontsize=8)
    axes[i].set_ylabel(TARGET, fontsize=8)
    sns.despine(ax=axes[i])

for j in range(len(top_features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle(f'Top Features vs {TARGET}  (* p<0.05, ** p<0.01, *** p<0.001)', y=1.01)
plt.tight_layout()
plt.show()

## 9. Summary & Modelling Plan

### Answers to Research Questions

| # | Question | Finding |
|---|---|---|
| 1 | Target structure | `stage` ∈ {0, 1, 2, 3} — ordinal, 4 classes. Mean=1.88, std=1.24. Treat as regression (RMSE-based) or ordinal classification. |
| 2 | Clinical drivers | `fasting_glucose` (ρ=0.83), `HbA1c` (ρ=0.72), `BMI` (ρ=0.71), `HOMA_IR` (ρ=0.70) — core glycaemic biomarkers dominate. |
| 3 | Non-linearity | Divergence up to 0.17 (e.g. `fasting_glucose`: Pearson=0.67, Spearman=0.83) → strong non-linear signal → tree models expected to outperform Ridge. |
| 4 | Data quality | 0 missing in main features. `onset_year` has 43% missing (216k/496k) — drop. `alive`=1.0 for all rows — constant, drop. |
| 5 | Feature redundancy | `HbA1c` and `fasting_glucose` ρ≈0.85 (both measure glycaemic control) → multicollinearity risk for Ridge; LightGBM handles automatically. |

---

### ⚠ Leakage Warning — Critical Finding

The following columns have Spearman ρ > 0.84 with `stage` and are **likely derived from the target** or from future information:

| Column | Spearman ρ | Problem |
|---|---|---|
| `diabetes_onset` | 0.919 | Binary flag for diabetes diagnosis — encodes the outcome |
| `glucose_prev` | 0.903 | Previous glucose — future relative to some rows |
| `future_diabetes_5yr` | 0.872 | Explicitly future label |
| `future_diabetes_risk` | 0.845 | Risk score derived from outcome |
| `risk_score` | 0.837 | Composite score likely including stage |

**These must be excluded from the feature set before modelling.** Including them inflates R² artificially and makes the model useless in deployment (you wouldn't have future labels at prediction time).

---

### Modelling Plan

**Target:** `stage` ∈ {0,1,2,3}  
**Features (clean set, leakage removed):** `fasting_glucose`, `HbA1c`, `BMI`, `HOMA_IR`, `insulin`, `age`, `triglycerides`, `HDL`, `LDL`, `blood_pressure`, `daily_calories`, `sugar_intake`, `sleep_hours`, `stress_level`, `sedentary_hours`, `family_history`, `on_medication`, `smoking`, `alcohol`, `bmi_prev`  
**Drop:** `patient_id`, `alive`, `onset_year`, `diabetes_onset`, `glucose_prev`, `future_diabetes_5yr`, `future_diabetes_risk`, `risk_score`  
**Baseline:** DummyRegressor (mean)  
**Linear:** Ridge  
**Tree:** LightGBM — non-linearity confirmed by divergence analysis  
**Evaluation:** RMSE + Spearman ρ; if treating as classification → macro-AUC  
**Validation:** 5-Fold CV (i.i.d. cross-sectional data, no temporal ordering needed)
